# Mini CNN on Synthetic Digit-like Data

A small CNN on **`DummyDataGenerator`** images — no downloads required.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic 28×28 grayscale images and 10 class labels."""
    def __init__(self, n_samples=256, n_classes=10, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, 1, 28, 28, generator=g)
        # add blob-like structure
        for i in range(n_samples):
            cx, cy = torch.randint(6, 22, (2,), generator=g)
            self.X[i, 0, cx-3:cx+3, cy-3:cy+3] += 2.0
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

class ImageDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class MiniCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(16 * 7 * 7, n_classes)
    def forward(self, x):
        x = self.conv(x)
        return self.fc(x.flatten(1))


## 1. Load synthetic data

In [ ]:
gen = DummyDataGenerator(n_samples=128)
ds = ImageDataset(gen.X, gen.y)
loader = DataLoader(ds, batch_size=16, shuffle=True)
bx, by = next(iter(loader))
print(f"batch: {bx.shape}, labels: {by.shape}")

## 2. Sample images grid

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.ravel(), range(8)):
    ax.imshow(gen.X[i, 0].numpy(), cmap='gray')
    ax.set_title(f'label {gen.y[i].item()}'); ax.axis('off')
plt.suptitle('Synthetic 28×28 images'); plt.tight_layout(); plt.show()

## 3. Train mini CNN

In [ ]:
model = MiniCNN()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
losses = []
for epoch in range(5):
    for bx, by in loader:
        opt.zero_grad()
        loss = F.cross_entropy(model(bx), by)
        loss.backward(); opt.step()
        losses.append(loss.item())
print(f"final loss: {losses[-1]:.3f}")

## 4. Training loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color='steelblue')
ax.set_xlabel('step'); ax.set_ylabel('loss')
ax.set_title('MiniCNN training loss')
plt.tight_layout(); plt.show()

## 5. Visualize first-layer filters

In [ ]:
filters = model.conv[0].weight.detach()  # 8 filters
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.ravel(), range(8)):
    ax.imshow(filters[i, 0].numpy(), cmap='RdBu_r')
    ax.set_title(f'filter {i}'); ax.axis('off')
plt.suptitle('Learned conv1 filters'); plt.tight_layout(); plt.show()